# exp006_hard_well_router_diagnostic inference

Diagnostic-only experiment; no submission is produced.


## Contents

1. Setup and paths
2. Diagnostic-only guard


## 1. Setup, Configuration, And Selected Variant


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import pandas as pd

from baseline import (
    HORIZONTAL_SUFFIX,
    active_feature_columns,
    build_drift_feature_frame,
    config_get,
    drift_strategy,
    fit_variant_model_from_files,
    gr_gate_weight,
    gr_gating_enabled,
    optional_positive_int,
    predict_variant_drift,
    primary_strategy,
    well_id_from_path,
)
from settings import EXPERIMENT_NAME, ExperimentPaths, deep_merge, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None


def apply_selected_variant(base_config: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any] | None]:
    selected_name = str(config_get(base_config, "ablation.selected_variant", "") or "")
    if not selected_name:
        return base_config, None

    variants = config_get(base_config, "ablation.variants", [])
    if not isinstance(variants, list):
        raise ValueError("ablation.variants must be a list when selected_variant is set")
    for variant in variants:
        if not isinstance(variant, dict):
            continue
        if str(variant.get("name")) != selected_name:
            continue
        overrides = variant.get("overrides") or {}
        if not isinstance(overrides, dict):
            raise ValueError(f"selected variant {selected_name} overrides must be a mapping")
        selected_config = deep_merge(base_config, overrides)
        return selected_config, variant
    raise ValueError(f"ablation.selected_variant not found: {selected_name}")


paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
base_config = load_config()
config, selected_variant = apply_selected_variant(base_config)
primary = primary_strategy(config)
drift_name = drift_strategy(config)
gating_enabled = gr_gating_enabled(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Selected variant:", selected_variant.get("name") if selected_variant else "base_config")
print("Primary strategy:", primary)
print("Drift strategy:", drift_name)
print("Feature set:", config_get(config, "model.feature_set", "all"))
print("Feature columns:", len(active_feature_columns(config)))
print("GR gating:", gating_enabled)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)


## 2. Diagnostic-only guard


In [ ]:
raise RuntimeError(
    "exp006_hard_well_router_diagnostic is diagnostic-only and does not "
    "produce submission.csv. Implement the selected router in the next experiment."
)
